# Qwen-Image 2.1 (GGUF) — Headless ComfyUI on Colab

This notebook runs **Qwen-Image 2.1** in pure **headless mode** without any UI overhead. It connects directly with your Google Drive for instant model caching, reference image loading, and automatic output saving.

**Workflow Structure:**
- **Cell 1**: Connect to Drive (`/Qwen-Image`), clone repository scripts, and install headless dependencies.
- **Cell 2**: **Multi-Developer Model Discovery & Selection**: Scans Drive and community providers (KasugaiSakura, Unsloth, AtomicChat, Comfy-Org), displays catalog, and syncs chosen model.
- **Cell 3**: **Launch ComfyUI & Pre-load VRAM**: Boots headless ComfyUI server, warms up model in GPU VRAM, and catalogs reference images (`<image1>`, `<image2>`, ...).
- **Cell 4**: **Configure Generation**: Set prompt, dimensions, steps, CFG, seed, and optional reference image editing (`<imageN>`).
- **Cell 5**: **Runner**: Initiates image generation, waits until complete, renders preview inline, and automatically saves output to `Drive/Qwen-Image/output`.


### Cell 1: Environment Setup, Google Drive Connection & Script Fetch
1. Connect to Google Drive and initialize root folder `/content/drive/MyDrive/Qwen-Image`.
2. Fetch the latest codebase scripts from GitHub repository.
3. Install ComfyUI and required dependencies headlessly.

In [ ]:
# --- 1. Connect to Google Drive & Create Folder Structure ---
import os, sys
from pathlib import Path

print(">> [1/3] Connecting to Google Drive...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print(">> Not in Colab environment (skipping drive.mount)")

DRIVE_ROOT = Path("/content/drive/MyDrive/Qwen-Image")
INPUT_DIR = DRIVE_ROOT / "input"
OUTPUT_DIR = DRIVE_ROOT / "output"
MODELS_DIR = DRIVE_ROOT / "models"
DIFFUSION_MODELS_DIR = MODELS_DIR / "Diffusion Models"
TEXT_ENCODERS_DIR = MODELS_DIR / "Text Encoders"
VAE_DIR = MODELS_DIR / "VAE"

for folder in [DRIVE_ROOT, INPUT_DIR, OUTPUT_DIR, MODELS_DIR, DIFFUSION_MODELS_DIR, TEXT_ENCODERS_DIR, VAE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print(f">> Google Drive root folder ready at: {DRIVE_ROOT}")

# --- 2. Download / Update Scripts from GitHub Repo ---
print("\n>> [2/3] Fetching latest scripts from GitHub repository...")
REPO_URL = "https://github.com/greencube-dev-boop/qwen-image-headless.git"
PROJECT_DIR = "/content/qwen21"

if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    !git -C {PROJECT_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# --- 3. Install Required Libraries & ComfyUI (Headless) ---
print("\n>> [3/3] Setting up ComfyUI and installing dependencies...")
!bash setup_colab.sh
print("\n>> Cell 1 Complete! Ready for Cell 2.")


### Cell 2: Multi-Developer Model Discovery & Selection
4. Scans Google Drive (`Drive/Qwen-Image/models/Diffusion Models`), local cache, and community providers (KasugaiSakura, Unsloth, AtomicChat, Comfy-Org).
5. Displays an interactive catalog table showing available models, quantization types, sizes, and cache status.
6. Selects and downloads your chosen model (fast-copies if in Drive, or downloads from HF and backs up to Drive).

In [ ]:
import os, sys
from IPython.display import display, HTML
from download_models import (
    build_model_catalog,
    format_model_catalog_table,
    resolve_model_choice,
    sync_selected_model
)

# --- Multi-Developer Model Discovery & Selection Settings ---
# Enter model ID number [1..N], filename, quant tag (e.g. Q4_K_M, Q8_0), or custom "repo_id:filename"
MODEL_CHOICE = "1"        #@param {type:"string"}
TEXT_ENCODER = "int8"     #@param ["int8", "bf16", "w4a8"]
INTERACTIVE_SELECT = False #@param {type:"boolean"}   (enable to prompt interactively via console)

DRIVE_ROOT = "/content/drive/MyDrive/Qwen-Image"

print(">> [Step 4] Discovering compatible models across Google Drive & community developers...")
catalog = build_model_catalog(drive_root=DRIVE_ROOT, comfy_dir="ComfyUI", fetch_online=True)

# Display visual interactive catalog table
display(HTML(format_model_catalog_table(catalog, as_html=True)))

# Resolve selection
if INTERACTIVE_SELECT:
    print(f">> Available IDs: 1 to {len(catalog)}")
    user_val = input(f"Enter model number [1-{len(catalog)}] (default 1): ").strip()
    selected_entry = resolve_model_choice(user_val or "1", catalog)
else:
    selected_entry = resolve_model_choice(MODEL_CHOICE, catalog)

print(f"\n>> Selected Model: [{selected_entry.get('id', '?')}] {selected_entry['filename']}")
print(f"   Developer: {selected_entry['developer']} | Quant: {selected_entry['quant']} | Size: ~{selected_entry['size_gb']} GB")
print(f"   Source: {selected_entry['source']} (Status: {selected_entry['status']})")

# Sync chosen model (Drive hit -> fast copy; HF -> download & backup to Drive)
SELECTED_DIFFUSION_FILE = sync_selected_model(
    entry=selected_entry,
    drive_root=DRIVE_ROOT,
    comfy_dir="ComfyUI",
    text_encoder=TEXT_ENCODER,
    skip_vae=False
)
SELECTED_MODEL_ENTRY = selected_entry
print(f"\n>> Cell 2 Complete! Model `{SELECTED_DIFFUSION_FILE}` is synced and ready.")


### Cell 3: Launch ComfyUI Server, Load to GPU VRAM & Input Images Table
7. Launches ComfyUI server headlessly on port 8188 with `--lowvram`.
8. Pre-loads the selected model weights into GPU VRAM (warming up the GPU).
9. Scans `Drive/Qwen-Image/input` and displays reference images in an ID table (`<image1>`, `<image2>`, ...).

In [ ]:
import os, sys, base64
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display, HTML
from generate import load_models_to_vram

# --- 5. Launch ComfyUI & Load Selected Model into GPU VRAM ---
LOWVRAM = False  #@param {type:"boolean"}   (enable for 15GB T4 GPUs)

if "SELECTED_DIFFUSION_FILE" not in globals():
    raise RuntimeError(">> Please run Cell 2 first to select and sync a model.")

print(f">> [Step 5] Launching ComfyUI server and loading `{SELECTED_DIFFUSION_FILE}` into GPU VRAM...")
load_models_to_vram(model=SELECTED_DIFFUSION_FILE, lowvram=LOWVRAM, comfy_dir="ComfyUI")

# --- 6. Load Input Images from Drive/Qwen-Image/input as a Table (<image1>, <image2>, ...) ---
print("\n>> [Step 6] Scanning Drive/Qwen-Image/input for reference images...")

def load_input_images_table(input_dir="/content/drive/MyDrive/Qwen-Image/input"):
    input_path = Path(input_dir)
    input_path.mkdir(parents=True, exist_ok=True)
    valid_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    files = sorted([p for p in input_path.iterdir() if p.suffix.lower() in valid_exts and not p.name.startswith(".")],
                   key=lambda p: p.name.lower())
    
    registry = {}
    if not files:
        html = f"""
        <div style="padding: 12px; border: 1px dashed #f57c00; background: #fff8e1; border-radius: 6px; font-family: sans-serif;">
            <b>No reference images found in:</b> <code>{input_dir}</code><br>
            To use image references, upload images (PNG, JPG, WEBP) to this Google Drive folder and re-run this cell.
        </div>
        """
        display(HTML(html))
        return registry
    
    rows = []
    for i, file_path in enumerate(files, start=1):
        tag = f"<image{i}>"
        registry[tag] = str(file_path)
        try:
            with PILImage.open(file_path) as im:
                w, h = im.size
                thumb = im.copy()
                thumb.thumbnail((120, 120))
                buf = BytesIO()
                thumb.save(buf, format="PNG")
                b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
                thumb_html = f'<img src="data:image/png;base64,{b64}" style="max-height: 80px; max-width: 100px; border-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.25);">'
        except Exception:
            w, h = "?", "?"
            thumb_html = "(no preview)"
            
        sz_mb = file_path.stat().st_size / (1024 * 1024)
        sz_str = f"{sz_mb:.2f} MB" if sz_mb >= 1.0 else f"{sz_mb * 1024:.0f} KB"
        rows.append(f"""
            <tr style="border-bottom: 1px solid #e0e0e0; text-align: center;">
                <td style="padding: 8px; font-weight: bold; color: #1a73e8; font-size: 14px;"><code>{tag}</code></td>
                <td style="padding: 6px;">{thumb_html}</td>
                <td style="padding: 8px; text-align: left; font-family: monospace;">{file_path.name}</td>
                <td style="padding: 8px; color: #555;">{w} &times; {h}</td>
                <td style="padding: 8px; color: #555;">{sz_str}</td>
            </tr>
        """)
        
    table_html = f"""
    <div style="font-family: sans-serif; margin-top: 10px;">
        <h4 style="margin: 0 0 8px 0; color: #202124;">Input Reference Images ({len(files)} loaded from Drive):</h4>
        <table style="border-collapse: collapse; width: 100%; max-width: 800px; background: white; border: 1px solid #ddd; border-radius: 6px; overflow: hidden;">
            <thead>
                <tr style="background: #f8f9fa; border-bottom: 2px solid #ddd; text-align: center; font-size: 13px;">
                    <th style="padding: 10px;">ID Tag</th>
                    <th style="padding: 10px;">Preview</th>
                    <th style="padding: 10px; text-align: left;">Filename</th>
                    <th style="padding: 10px;">Dimensions</th>
                    <th style="padding: 10px;">Size</th>
                </tr>
            </thead>
            <tbody>
                {''.join(rows)}
            </tbody>
        </table>
        <p style="font-size: 12px; color: #5f6368; margin-top: 6px;">Reference these directly in your prompt in Cell 4 using <code>&lt;image1&gt;</code>, <code>&lt;image2&gt;</code>, etc.</p>
    </div>
    """
    display(HTML(table_html))
    return registry

DRIVE_ROOT = "/content/drive/MyDrive/Qwen-Image"
INPUT_IMAGES = load_input_images_table(os.path.join(DRIVE_ROOT, "input"))
print("\n>> Cell 3 Complete! Model loaded in VRAM and input references cataloged.")


### Cell 4: Generation Configuration & Image Reference Parsing
10. Configure prompt, negative prompt, steps, CFG, seed, and resolution.
- If `IMAGE_REFERENCE = True`, any tags like `<image1>`, `<image2>` mentioned in the prompt are automatically extracted, mapped, and passed as reference images to the model.

In [ ]:
import re
from pathlib import Path

# --- 7. Configure Generation Settings ---
PROMPT = "Cinematic portrait of a woman in a red jacket on a rooftop at dusk, city bokeh, 85mm"  #@param {type:"string"}
NEGATIVE = ""            #@param {type:"string"}
IMAGE_REFERENCE = False  #@param {type:"boolean"}   (enable to use <imageN> tags from the table above)
WIDTH = 1024             #@param {type:"integer"}   (canvas width; in edit mode 0 = follow <image1>)
HEIGHT = 1024            #@param {type:"integer"}   (canvas height; in edit mode 0 = follow <image1>)
STEPS = 25               #@param {type:"integer"}
CFG = 1.0                #@param {type:"number"}
SEED = -1                #@param {type:"integer"}   (-1 for random seed)
COUNT = 1                #@param {type:"integer"}   (number of images to generate)
RESOLUTION = 1024        #@param {type:"integer"}   (reference images pixel budget in edit mode)

# Parse and validate image reference tags in the prompt
referenced_tags = re.findall(r"<image\d+>", PROMPT)
unique_tags = list(dict.fromkeys(referenced_tags))
effective_prompt = PROMPT
images_to_send = []

if IMAGE_REFERENCE:
    if not unique_tags:
        print(">> Warning: IMAGE_REFERENCE is True, but no <imageN> tags were found in PROMPT.")
        print(">> Mention tags like <image1>, <image2> from the table above in your prompt.")
    else:
        if "INPUT_IMAGES" not in globals() or not INPUT_IMAGES:
            raise RuntimeError(">> INPUT_IMAGES table is empty or Cell 3 has not been run. Please run Cell 3 first.")
        
        missing = [t for t in unique_tags if t not in INPUT_IMAGES]
        if missing:
            raise ValueError(f"Referenced tags {missing} not found in input table. Available: {list(INPUT_IMAGES.keys())}")
        
        images_to_send = [INPUT_IMAGES[t] for t in unique_tags]
        print(f">> Reference images detected ({len(unique_tags)}):\n")
        
        # Map tags to consecutive slots for ComfyUI: <image1>, <image2>, ...
        tag_map = {}
        for idx, orig_tag in enumerate(unique_tags, start=1):
            target_tag = f"<image{idx}>"
            tag_map[orig_tag] = target_tag
            print(f"   {orig_tag} ({Path(INPUT_IMAGES[orig_tag]).name}) -> model slot {target_tag}")
            
        effective_prompt = re.sub(r"<image\d+>", lambda m: tag_map.get(m.group(0), m.group(0)), effective_prompt)

active_model = SELECTED_DIFFUSION_FILE if "SELECTED_DIFFUSION_FILE" in globals() else "qwen-image-2.1-UC-Q4_K_M.gguf"
GEN_CONFIG = {
    "prompt": PROMPT,
    "effective_prompt": effective_prompt,
    "negative": NEGATIVE,
    "image_reference": IMAGE_REFERENCE,
    "referenced_tags": unique_tags,
    "images": images_to_send,
    "width": WIDTH,
    "height": HEIGHT,
    "steps": STEPS,
    "cfg": CFG,
    "seed": SEED,
    "count": COUNT,
    "resolution": RESOLUTION,
    "model": active_model
}

print("\n>> Configuration locked and ready for Cell 5 Runner:")
print(f"   Model: {active_model}")
print(f"   Mode: {'IMAGE REFERENCE (Edit)' if IMAGE_REFERENCE else 'TEXT TO IMAGE'}")
print(f"   Resolution: {WIDTH}x{HEIGHT} | Steps: {STEPS} | CFG: {CFG} | Count: {COUNT} | Seed: {SEED}")


### Cell 5: Runner — Execute Generation & Save Output
11. Initiates generation with the configured parameters, monitors progress until finished, displays inline preview, and automatically saves the output images into `Drive/Qwen-Image/output`.

In [ ]:
import time, shutil
from pathlib import Path
from IPython.display import display, Image
from generate import run, edit, ComfyError

if "GEN_CONFIG" not in globals():
    raise RuntimeError(">> Please run Cell 4 first to lock in GEN_CONFIG.")

cfg = GEN_CONFIG
out_local = Path("/content/qwen_out")
out_local.mkdir(exist_ok=True)
drive_output_dir = Path("/content/drive/MyDrive/Qwen-Image/output")
drive_output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print(f">> Initiating generation job at {time.strftime('%H:%M:%S')}...")
print(f">> Model: {cfg['model']}")
print(f">> Mode: {'Image Reference (Edit)' if cfg['image_reference'] else 'Text to Image'}")
print(f">> Prompt: {cfg['prompt']}")
if cfg["image_reference"]:
    print(f">> Reference images fed to model ({len(cfg['images'])}): {', '.join(cfg['referenced_tags'])}")
print("=" * 65)

start_time = time.time()
files = []
used_seed = None

try:
    if cfg["image_reference"]:
        if not cfg["images"]:
            raise ValueError("IMAGE_REFERENCE is True but no valid reference images were found in the prompt.")
        files, used_seed = edit(
            prompt=cfg["effective_prompt"],
            images=cfg["images"],
            negative=cfg["negative"],
            resolution=cfg["resolution"],
            width=cfg["width"] if cfg["width"] > 0 and cfg["height"] > 0 else None,
            height=cfg["height"] if cfg["width"] > 0 and cfg["height"] > 0 else None,
            steps=cfg["steps"],
            cfg=cfg["cfg"],
            seed=None if cfg["seed"] < 0 else cfg["seed"],
            count=cfg["count"],
            model=cfg["model"],
            out=str(out_local)
        )
    else:
        files, used_seed = run(
            prompt=cfg["prompt"],
            negative=cfg["negative"],
            width=cfg["width"],
            height=cfg["height"],
            steps=cfg["steps"],
            cfg=cfg["cfg"],
            seed=None if cfg["seed"] < 0 else cfg["seed"],
            count=cfg["count"],
            model=cfg["model"],
            out=str(out_local)
        )

    elapsed = time.time() - start_time
    print(f"\n>> Generation finished in {elapsed:.1f}s | Seed: {used_seed}")

    # Save images to Google Drive output folder
    saved_to_drive = []
    for f in files:
        src = Path(f)
        dst = drive_output_dir / src.name
        counter = 1
        while dst.exists():
            dst = drive_output_dir / f"{src.stem}_{counter}{src.suffix}"
            counter += 1
        shutil.copy2(src, dst)
        saved_to_drive.append(dst)
        print(f">> Saved to Google Drive: {dst}")

    # Display images inline
    print("\n>> Output Image Preview:")
    for f in files:
        display(Image(f))

except ComfyError as e:
    print(f">> ComfyError during execution: {e}")
    if e.files:
        print(">> Partial images completed before error:")
        for f in e.files:
            display(Image(f))
except Exception as e:
    print(f">> Execution error: {e}")
    raise
